# Suite PTH — Paths

`Paths.of(store)` names every object a store's roots reach: a schema of a store of schemas by its name, a singleton by
its global name, and any other object by the first route from them, each step `/adjacency[key]`, its key the entry's
values that a unique constraint declares with the object's own link, else its position. A path survives changes that
do not touch the route to the object.

In [ ]:
from mbse.Schemas.Framework import Paths, Proxies, Reflection, Schemas as S, Stores
from support import raises, text

Listed = S.OfRelation.Builder().name("Listed").links("directory", "contact").create()
Phones = S.OfRelation.Builder().name("Phones").links("owner", "phone").properties(text("label")).unique(
    "owner", "label").unique("phone").create()  # the second does not hold the owner
Slots = S.OfRelation.Builder().name("Slots").links("owner", "phone").properties(text("slot", int), text("on", bool),
                                                                                 text("amount", float)).unique(
    "owner", "amount").unique("owner", "slot", "on").create()  # an amount is not written
Enrolled = S.OfRelation.Builder().name("Enrolled").links("student", "course", "term").unique("student").unique(
    "student", "term").create()  # neither holds a property
Directory = S.OfObject.Builder().name("Directory").ref().singleton("book.Directory").relations(
    lambda r: r.name("contacts").of(Listed).me("directory")).create()
Contact = S.OfObject.Builder().name("Contact").ref().properties(text("name")).relations(
    lambda r: r.name("directories").of(Listed).me("contact"), lambda r: r.name("phones").of(Phones).me("owner"),
    lambda r: r.name("slots").of(Slots).me("owner"), lambda r: r.name("enrolled").of(Enrolled).me("student")).create()
Phone = S.OfObject.Builder().name("Phone").ref().properties(text("number")).create()
Course = S.OfObject.Builder().name("Course").ref().properties(text("title")).create()
Archive = S.OfObject.Builder().name("Archive").ref().singleton("an.Archive").relations(
    lambda r: r.name("contacts").of(Listed).me("directory")).create()


def book():
    """A store of the schemas, its directory listing Ann, with two phones by label, a slot and an enrollment."""
    store = Proxies.OfStore()
    for schema in (Directory, Archive, Contact, Phone, Course, Listed, Phones, Slots, Enrolled):
        store.register(schema)
    one, two, three, four = (store.Phone().number(n).create() for n in "1234")
    math, fall = store.Course().title("math").create(), store.Course().title("fall").create()
    ann = store.Contact().name("ann").phones(lambda e: e.phone(one).label("home")).phones(
        lambda e: e.phone(two).label("work")).slots(lambda e: e.phone(three).slot(7).on(True).amount(0.5)).slots(lambda e: e.phone(four).slot(7).on(False)).enrolled(
        lambda e: e.course(math).term(fall)).create()
    store.Directory(store.singleton("book.Directory")).contacts(lambda e: e.contact(ann)).update()
    return store, ann

## PTH-01 · Singletons by their global names, other objects by route: keys by unique properties, else by position

In [ ]:
store, ann = book()
paths = Paths.of(store)
assert paths.of(store.singleton("book.Directory")) == "book.Directory" and paths.of(ann) == "book.Directory/contacts[0]"
phones = {p.number: paths.of(p) for p in store.extent("Phone")}
assert phones == {"1": 'book.Directory/contacts[0]/phones[label="home"]', "2": 'book.Directory/contacts[0]/phones[label="work"]',
                  "3": "book.Directory/contacts[0]/slots[on=true,slot=7]", "4": "book.Directory/contacts[0]/slots[on=false,slot=7]"}  # the first constraint a path writes
courses = {c.title: paths.of(c) for c in store.extent("Course")}
assert courses == {"math": "book.Directory/contacts[0]/enrolled[0].course", "fall": "book.Directory/contacts[0]/enrolled[0].term"}
assert paths.of(store.singleton("an.Archive")) == "an.Archive" and paths.find("book.Directory/contacts[0]") is ann
assert Paths.Paths(store).of(ann) == paths.of(ann)
with raises(LookupError, match="no object at 'nowhere'"):
    paths.find("nowhere")
with raises(LookupError, match="no root reaches the object"):
    paths.of(store.Contact().name("loose").create())

## PTH-02 · A path survives changes elsewhere: a property set, an entry added after; a positional one before shifts

In [ ]:
store, ann = book()
before = Paths.of(store)
home = before.find('book.Directory/contacts[0]/phones[label="home"]')
store.Phone(home).number("9").update()
store.Contact(ann).phones(lambda e: e.phone(store.Phone().number("4").create()).label("cell")).update()
after = Paths.of(store)
assert after.of(home) == before.of(home) and after.of(ann) == before.of(ann)
bob = store.Contact().name("bob").create()
store.Directory(store.singleton("book.Directory")).contacts(lambda e: e.contact(bob)).update()
assert Paths.of(store).of(bob) == "book.Directory/contacts[1]" and Paths.of(store).of(ann) == "book.Directory/contacts[0]"
store.Archive(store.singleton("an.Archive")).contacts(lambda e: e.contact(bob)).update()
assert Paths.of(store).of(bob) == "an.Archive/contacts[0]"  # reached first from the root first in name order

## PTH-03 · A store of schemas names its schemas; combined stores name the schemas, then the singletons

In [ ]:
store, ann = book()
schemas = Reflection.of(store)
named = Paths.of(schemas)
assert named.of(Contact) == "Contact" and named.find("Phones") is Phones and named.of(Directory) == "Directory"
both = Stores.Combined(schemas, store)
combined = Paths.of(both)
assert (combined.of(Contact), combined.of(ann)) == ("Contact", "book.Directory/contacts[0]")
assert Paths.of(Stores.Combined(Stores.Combined(schemas), store)).of(Phone) == "Phone"  # combined again